<a href="https://colab.research.google.com/github/CarlosNobuaki/AULA1-Dados-Estruturados/blob/master/crawler_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crawler Milho CEPEA ESALQ (Synapse)

## Fonte: https://cepea.org.br/br

## Fluxo
1. Instalar dependencias
2. Fazer upload do XLS base (opcional, mas recomendado)
3. Scrapar dados diarios do CEPEA
4. Inicializar ou atualizar `cotacao_milho.xlsx`
5. Fazer download do arquivo final

In [10]:
# Dependencias para Colab
!pip -q install curl_cffi beautifulsoup4 pandas openpyxl xlrd ipywidgets

In [19]:
#É uma biblioteca padrão do Python que permite ao seu script interagir com o sistema operacional.
import os
#É uma biblioteca especializada em manipulação de dados (Dataframes)
import pandas as pd
#É uma biblioteca de elementos visuais interativos para o Jupyter.
import ipywidgets as widgets
#É uma bibliotaca de limpa e traduz todo o código da página, permitindo a raspagem de dados importantes pré requisitados
from bs4 import BeautifulSoup
#É uma biblioteca que simula a navegação humana para obter os dados HTML da página
from curl_cffi import requests as cf
#É uma biblioteca de exibição, renderiza tabelas e outras coisas
from IPython.display import display, clear_output


# Variáveis
CEPEA_URL = 'https://cepea.org.br/br/indicador/milho.aspx'
XLS_BASE = '/content/cepea_base.xls'
XLS_SAIDA = '/content/cotacao_milho.xlsx'

print('Ambiente pronto.')

Ambiente pronto.


## Upload do XLS base (opcional)
Se voce tiver o XLS historico baixado do CEPEA, faca upload aqui.

Se nao enviar, o notebook continua e cria a aba `Media_Anual` vazia.

In [3]:
from google.colab import files

print('Selecione o XLS base do CEPEA (opcional).')
# Abre a janela de importação de arquivos
uploaded = files.upload()
# Verifica se existe arquivo, escolhe o primeiro e cria o caminho do arquivo na pasta temporária do colab
if len(uploaded) > 0:
    nome_arquivo = next(iter(uploaded.keys()))
    origem = f'/content/{nome_arquivo}'
    if origem != XLS_BASE:
        import shutil
        shutil.copy(origem, XLS_BASE)
    print(f'XLS base salvo como: {XLS_BASE}')
else:
    print('Nenhum XLS base enviado. Continuando sem medias anuais.')

Selecione o XLS base do CEPEA (opcional).


Saving cepea-consulta-20260315134332.xls to cepea-consulta-20260315134332.xls
XLS base salvo como: /content/cepea_base.xls


In [21]:
# Função de limpeza dos dados
def limpar_numero(serie: pd.Series) -> pd.Series:
    # Converte padrao brasileiro (1.234,56 e 0,59%) para float
    return pd.to_numeric(
        #tudo que chegar nessa função será convertido para string
        serie.astype(str)
              #remove todos os espaços em branco
             .str.replace(r'\s', '', regex=True)
             #---------------------------------------------------------------------------------
              #remove a porcentagem
             .str.replace('%', '', regex=False)
             #---------------------------------------------------------------------------------
              #remove o ponto de milhar
             .str.replace('.', '', regex=False)
             #---------------------------------------------------------------------------------
              #substitui a vírgula decimal pelo ponto decimal
             .str.replace(',', '.', regex=False)
             #---------------------------------------------------------------------------------
             #Remove tudo que NÃO for um dígito (\d), um ponto (.) ou um sinal de menos (\-)".
             #Isso elimina letras ou caracteres especiais acidentais que ainda possam estar lá.
             .str.replace(r'[^\d.\-]', '', regex=True),
        #Tudo que não estiver dentro das especificações acima será considerado NaN (not a number)
        errors='coerce',
    )

# Função de rapsgem da tabela
def scrape_tabela() -> pd.DataFrame:
    # Printa qual site esta sendo acessado
    print(f'Conectando: {CEPEA_URL}')
    # Utiliza a biblioteca (curl_cffi) para simular uma interação humana e captura o código HTML
    resp = cf.get(CEPEA_URL, impersonate='chrome120', timeout=30)
    # Se der erro de conexão, a execução para
    resp.raise_for_status()
    # Transforma o código HTML bruto em objeto python
    soup = BeautifulSoup(resp.text, 'html.parser')
    # Captura a primeita tag de tabela do HTML
    tabela = soup.find('table')
    if tabela is None:
        raise RuntimeError('Tabela nao encontrada na pagina do CEPEA.')

    linhas = []
    #Percorre cada linha (tr) da tabela encontrada.
    for tr in tabela.find_all('tr'):
        #Dentro de cada linha, pega o texto de cada célula (td). O strip=True remove espaços sobrando no início e fim.
        cols = [td.get_text(strip=True) for td in tr.find_all('td')]
        #Se tiver 5 colunas e o primeiro itém da coluna não estiver vazio
        if len(cols) == 5 and cols[0]:
            # salva os dados na variável 'linhas'
            linhas.append(cols)

    if not linhas:
        raise RuntimeError('Nenhuma linha de dados encontrada na tabela.')
    #Cria uma tabela inicial com os dados brutos e dá nome as colunas
    df = pd.DataFrame(
        linhas,
        columns=['data_str', 'valor_rs_str', 'var_dia_str', 'var_mes_str', 'valor_usd_str'],
    )
    #Cria um novo dataframe para armazedar os dados convertidos
    resultado = pd.DataFrame()
    #Converte o texto de data em objeto de data começando pelo dia (01/05/2026)
    resultado['data'] = pd.to_datetime(df['data_str'], dayfirst=True, errors='coerce')
    #Utiliza a função 'limpar_numeros' para padronizar os números
    resultado['valor_rs'] = limpar_numero(df['valor_rs_str'])
    resultado['var_dia_pct'] = limpar_numero(df['var_dia_str'])
    resultado['var_mes_pct'] = limpar_numero(df['var_mes_str'])
    resultado['valor_usd'] = limpar_numero(df['valor_usd_str'])

    return (
        resultado
        # remove valores vazios
        .dropna(subset=['data', 'valor_rs'])
        # organiza em ordem crescente as datas
        .sort_values('data')
        # refaz o index após o tratamento dos dados
        .reset_index(drop=True)
    )

#Importa os dados cepea_base.xls
def importar_base_xls() -> pd.DataFrame:
    #Verifica se não existe os dados
    if not os.path.exists(XLS_BASE):
        print('[AVISO] XLS base nao encontrado. Media_Anual sera criada vazia.')
        #Retorna um dataframe com os nomes das colunas e valores vazios
        return pd.DataFrame(columns=['ano', 'valor_rs_medio', 'valor_usd_medio'])
    #Faz a leitura no excel
    raw = pd.read_excel(XLS_BASE, header=None, engine='xlrd')

    header_row = 3
    #Percorre cada linha do dataframe até encontrar a string 'data' e armazena os valores como títulos
    for i, row in raw.iterrows():
        if any('data' in str(c).lower() for c in row.values):
            header_row = int(i)
            break
    #Agora lê o arquivo novamente, mas pulando as linhas inúteis do topo e começando direto nos títulos reais.
    df = pd.read_excel(XLS_BASE, header=header_row, engine='xlrd')
    #Limpa espaços em branco invisíveis nos nomes das colunas
    df.columns = [str(c).strip() for c in df.columns]

    #Função para encontra colunas mesmo com o nome um pouco diferente (minúsculas e maiúsculas)
    def achar(palavras):
        for p in palavras:
            for col in df.columns:
                if p.lower() in col.lower():
                    return col
        return None
    #Tenta achar a coluna de tempo
    col_ano = achar(['data', 'ano'])
    #Tenta achar a coluna em Reias
    col_rs = achar(['r$', 'vista r', 'reais'])
    #Tenta achar a coluna em dolllar
    col_usd = achar(['us$', 'dolar', 'dollar'])

    #Cria a tabela limpa
    resultado = pd.DataFrame()
    #Atribui as datas e se não encontrar as datas adiciona '?'
    resultado['ano'] = df[col_ano].astype(str).str.strip() if col_ano else '?'
    #Atribui o valor em reais
    resultado['valor_rs_medio'] = limpar_numero(df[col_rs]) if col_rs else float('nan')
    #Atribui o valor em dollar
    resultado['valor_usd_medio'] = limpar_numero(df[col_usd]) if col_usd else float('nan')
    #Apaga as linhas vazias e reorganiza o index
    return resultado.dropna(subset=['valor_rs_medio']).reset_index(drop=True)

#Compara o arquivo já existente com os novos dados e apaga duplicatas
def ler_diario_existente() -> pd.DataFrame:
    colunas = ['data', 'valor_rs', 'var_dia_pct', 'var_mes_pct', 'valor_usd']
    if not os.path.exists(XLS_SAIDA):
        return pd.DataFrame(columns=colunas)
    try:
        df = pd.read_excel(XLS_SAIDA, sheet_name='Diario', engine='openpyxl')
        df['data'] = pd.to_datetime(df['data'], errors='coerce')
        return df
    except Exception:
        return pd.DataFrame(columns=colunas)

#Essa função junta os dados e salva em arquivo excel, mas com duas abas 'Diario' e 'Media_Anual'
def salvar_xls(df_diario: pd.DataFrame, df_anual: pd.DataFrame) -> None:
    with pd.ExcelWriter(
        XLS_SAIDA,
        engine='openpyxl',
        date_format='DD/MM/YYYY',
        datetime_format='DD/MM/YYYY',
    ) as writer:
        df_diario.to_excel(writer, sheet_name='Diario', index=False)
        df_anual.to_excel(writer, sheet_name='Media_Anual', index=False)

## Execucao interativa com widgets
Use os controles abaixo para escolher o modo e executar o web scraper.

- `Atualizar`: adiciona apenas datas novas ao `cotacao_milho.xlsx`
- `Reimportar`: recria o arquivo com base no XLS historico + scraping

In [23]:
def executar_crawler(reimportar=False):
    #Verifica se você escolheu "Recriar arquivo" no menu ou se o arquivo simplesmente não existe ainda.
    if reimportar or not os.path.exists(XLS_SAIDA):
        print('--- Inicializando cotacao_milho.xlsx ---')
        #Carrega as médias históricas do Excel do CEPEA.
        df_anual = importar_base_xls()
        print(f'Media_Anual: {len(df_anual)} linha(s)')
        #ai ao site e baixa os preços mais recentes.
        df_diario = scrape_tabela()
        print(
            f'Diario: {len(df_diario)} linha(s) '
            f'({df_diario["data"].min().date()} -> {df_diario["data"].max().date()})'
        )

        salvar_xls(df_diario, df_anual)
        print(f'Arquivo criado: {XLS_SAIDA}')
        return
    #Verifica se você escolheu "Atualizar" no menu e se os dados são existentes
    print('--- Atualizando cotacao_milho.xlsx ---')
    df_existente = ler_diario_existente()
    #Se os dados estiverem vazios, repete os passos para carregar os dados
    if df_existente.empty:
        print('Aba Diario vazia ou ausente. Recriando arquivo...')
        df_anual = importar_base_xls()
        df_diario = scrape_tabela()
        salvar_xls(df_diario, df_anual)
        return
    #Cria uma lista (set) com todas as datas que você já tem salvas. O normalize() remove as horas, deixando apenas o dia/mês/ano.
    datas_existentes = set(df_existente['data'].dt.normalize())
    ultima_data = df_existente['data'].max()
    print(f'Linhas atuais: {len(df_existente)} | ultima data: {ultima_data.date()}')
    #Baixa os dados atuais
    df_novo = scrape_tabela()
    #compara os dados do site com os dados do arquivo e mantém apenas as datas que não tem.
    df_novo_filtrado = df_novo[
        ~df_novo['data'].dt.normalize().isin(datas_existentes)
    ].copy()

    if df_novo_filtrado.empty:
        print('Nenhuma data nova encontrada. Arquivo ja atualizado.')
        return

    try:
        df_anual = pd.read_excel(XLS_SAIDA, sheet_name='Media_Anual', engine='openpyxl')
    except Exception:
        df_anual = importar_base_xls()
    #Junta o que já tinha com os novos dados do dia
    df_combinado = (
        pd.concat([df_existente, df_novo_filtrado], ignore_index=True)
        .drop_duplicates(subset=['data'])
        .sort_values('data')
        .reset_index(drop=True)
    )
    salvar_xls(df_combinado, df_anual)
    print(f'+{len(df_novo_filtrado)} linha(s) adicionada(s).')
    print(f'Total Diario: {len(df_combinado)}')

# Cria a caixa de seleção
modo_widget = widgets.Dropdown(
    options=[('Atualizar (incremental)', False), ('Reimportar (recriar arquivo)', True)],
    value=False,
    description='Modo:',
)
#Cria os botões
botao_widget = widgets.Button(description='Executar crawler', button_style='success')
saida_widget = widgets.Output()

def _ao_clicar(_):
    with saida_widget:
        clear_output(wait=True)
        try:
            executar_crawler(reimportar=modo_widget.value)
            print('Execucao finalizada com sucesso.')
        except Exception as exc:
            print(f'Erro durante execucao: {exc}')
#Executa
botao_widget.on_click(_ao_clicar)
display(widgets.VBox([modo_widget, botao_widget, saida_widget]))

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# Visualizacao rapida
df_diario_final = pd.read_excel(XLS_SAIDA, sheet_name='Diario', engine='openpyxl')
df_anual_final = pd.read_excel(XLS_SAIDA, sheet_name='Media_Anual', engine='openpyxl')

print('Diario - ultimas 5 linhas:')
display(df_diario_final)

print('Media_Anual - ultimas 5 linhas:')
display(df_anual_final)

Diario - ultimas 5 linhas:


,data,valor_rs,var_dia_pct,var_mes_pct,valor_usd
0,2026-02-25,69.28,0.20,4.81,13.51
1,2026-02-26,69.28,0.00,4.81,13.47
2,2026-02-27,69.53,0.36,5.19,13.53
3,2026-03-02,69.75,0.32,0.32,13.49
4,2026-03-03,69.98,0.33,0.65,13.28
5,2026-03-04,70.23,0.36,1.01,13.46
6,2026-03-05,70.24,0.01,1.02,13.30
7,2026-03-06,70.56,0.46,1.48,13.44
8,2026-03-09,70.63,0.10,1.58,13.66
9,2026-03-10,71.09,0.65,2.24,13.77


Media_Anual - ultimas 5 linhas:


,ano,valor_rs_medio,valor_usd_medio
0,2024,63.37,11.79
1,2025,71.70,12.82
2,2026,68.44,13.01


In [18]:
# Download do resultado
from google.colab import files
files.download(XLS_SAIDA)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>